In [1]:
import lmdb
import six
from PIL import Image

In [4]:
from pathlib import Path

def find_lmdb_dirs(root_dir):
    root_dir = Path(root_dir)
    lmdb_dirs = []
    for data_file in root_dir.rglob("data.mdb"):
        if (data_file.parent / "lock.mdb").exists():
            lmdb_dirs.append(data_file.parent)
    return sorted(set(lmdb_dirs))

# TEXTZOOM_ROOT = r"E:\TT"
EXTZOOM_ROOT = "path/to/mdbfiles"

lmdb_dirs = find_lmdb_dirs(TEXTZOOM_ROOT)

print("找到的 LMDB 文件夹：")
for d in lmdb_dirs:
    print(d)

找到的 LMDB 文件夹：
E:\TT\test\easy
E:\TT\test\hard
E:\TT\test\medium
E:\TT\train1
E:\TT\train2


In [5]:
from pathlib import Path
from io import BytesIO
import lmdb
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

def lmdb_image_to_pil(txn, key):
    value = txn.get(key)
    if value is None:
        return None
    return Image.open(BytesIO(value)).convert("RGB")

def export_one_lmdb(lmdb_dir, out_dir):
    lmdb_dir = Path(lmdb_dir)
    out_dir = Path(out_dir)

    lr_dir = out_dir / "lr"
    hr_dir = out_dir / "hr"
    lr_dir.mkdir(parents=True, exist_ok=True)
    hr_dir.mkdir(parents=True, exist_ok=True)

    env = lmdb.open(
        str(lmdb_dir),
        readonly=True,
        lock=False,
        readahead=False,
        meminit=False,
        max_readers=1
    )

    rows = []

    with env.begin(write=False) as txn:
        raw_n = txn.get(b"num-samples")
        if raw_n is None:
            raise ValueError(f"{lmdb_dir} 里没有找到 num-samples")

        n_samples = int(raw_n.decode("utf-8"))

        for idx in tqdm(range(1, n_samples + 1), desc=str(lmdb_dir)):
            label_key = f"label-{idx:09d}".encode()
            lr_key = f"image_lr-{idx:09d}".encode()
            hr_key = f"image_hr-{idx:09d}".encode()

            raw_label = txn.get(label_key)
            if raw_label is None:
                continue

            label = raw_label.decode("utf-8", errors="replace")

            lr_img = lmdb_image_to_pil(txn, lr_key)
            hr_img = lmdb_image_to_pil(txn, hr_key)

            file_stem = f"{idx:09d}"
            lr_path = ""
            hr_path = ""

            if lr_img is not None:
                lr_file = lr_dir / f"{file_stem}.png"
                lr_img.save(lr_file)
                lr_path = str(lr_file)

            if hr_img is not None:
                hr_file = hr_dir / f"{file_stem}.png"
                hr_img.save(hr_file)
                hr_path = str(hr_file)

            rows.append({
                "index": idx,
                "label": label,
                "lr_path": lr_path,
                "hr_path": hr_path
            })

    env.close()

    df = pd.DataFrame(rows)
    csv_path = out_dir / "labels.csv"
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")

    print(f"\n导出完成: {lmdb_dir}")
    print(f"样本数: {len(df)}")
    print(f"CSV: {csv_path}")

    return df

TEXTZOOM_ROOT = Path(r"E:\TT")
EXPORT_ROOT = Path(r"E:\TT_exported")
EXPORT_ROOT.mkdir(parents=True, exist_ok=True)

all_dfs = []

for lmdb_dir in find_lmdb_dirs(TEXTZOOM_ROOT):
    rel_path = lmdb_dir.relative_to(TEXTZOOM_ROOT)
    out_dir = EXPORT_ROOT / rel_path
    df = export_one_lmdb(lmdb_dir, out_dir)
    df["split"] = str(rel_path)
    all_dfs.append(df)

merged_df = pd.concat(all_dfs, ignore_index=True)
merged_csv = EXPORT_ROOT / "all_labels.csv"
merged_df.to_csv(merged_csv, index=False, encoding="utf-8-sig")

print("\n全部导出完成")
print("总样本数:", len(merged_df))
print("总表路径:", merged_csv)

C:\Users\Eric Zhang\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
E:\TT\test\easy: 100%|████████████████████████████████████████████████████████████| 1619/1619 [00:01<00:00, 861.77it/s]



导出完成: E:\TT\test\easy
样本数: 1619
CSV: E:\TT_exported\test\easy\labels.csv


E:\TT\test\hard: 100%|████████████████████████████████████████████████████████████| 1343/1343 [00:06<00:00, 204.90it/s]



导出完成: E:\TT\test\hard
样本数: 1343
CSV: E:\TT_exported\test\hard\labels.csv


E:\TT\test\medium: 100%|██████████████████████████████████████████████████████████| 1411/1411 [00:01<00:00, 721.70it/s]



导出完成: E:\TT\test\medium
样本数: 1411
CSV: E:\TT_exported\test\medium\labels.csv


E:\TT\train1: 100%|█████████████████████████████████████████████████████████████| 14573/14573 [00:50<00:00, 287.96it/s]



导出完成: E:\TT\train1
样本数: 14573
CSV: E:\TT_exported\train1\labels.csv


E:\TT\train2: 100%|███████████████████████████████████████████████████████████████| 2794/2794 [00:03<00:00, 734.64it/s]


导出完成: E:\TT\train2
样本数: 2794
CSV: E:\TT_exported\train2\labels.csv

全部导出完成
总样本数: 21740
总表路径: E:\TT_exported\all_labels.csv
